# Gold Layer — Brand Analytics

## Business Definitions

| Concept | Definition |
|---|---|
| Revenue (net) | `l_extendedprice * (1 - l_discount)`, excluding tax |
| List price | `p_retailprice`, a unit price |
| Supply cost | `ps_supplycost` for the exact product-supplier pair |
| List-price margin | `p_retailprice - ps_supplycost` |
| Actual unit price | net revenue / `l_quantity` |
| Price deviation | `abs(actual_unit_price - p_retailprice) / p_retailprice` |
| Reporting date | `o_orderdate` |
| Category | full `p_type` value |
| Market share | brand category revenue / total category revenue |
| Affected product | at least one sale line with deviation > 0.15 |
| Average margin | supplier-offer average per product, then equal-product average |

A brand can belong to multiple categories. All money is decimal; rounding is presentation only.
Unknown margins are null, not zero.
Run `05_run_member2` for ordered execution. SQL alert creation remains a workspace setup step.


In [ ]:
import re

dbutils.widgets.text("catalog", "workspace", "Target catalog")  # portability: override per workspace
CATALOG = dbutils.widgets.get("catalog").strip()
BRONZE_SCHEMA = "bda_bronze"
SILVER_SCHEMA = "bda_silver"
GOLD_SCHEMA = "bda_gold"

def _valid_ident(s):
    if not re.match(r'^[A-Za-z_][A-Za-z0-9_]*$', s):
        raise ValueError(f"invalid identifier: {s}")

for s in (CATALOG, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA):
    _valid_ident(s)

spark.sql(f"USE CATALOG {CATALOG}")
print(f"catalog={CATALOG}  bronze={BRONZE_SCHEMA}  silver={SILVER_SCHEMA}  gold={GOLD_SCHEMA}")

In [ ]:
import uuid, time

run_id = str(uuid.uuid4())
run_ts = time.strftime("%Y-%m-%dT%H:%M:%S")
_errors = []
S = SILVER_SCHEMA

def _check(name, query, critical=True):
    try:
        n = spark.sql(query).collect()[0][0]
        if n != 0:
            _errors.append(f"[{'CRITICAL' if critical else 'WARN'}] {name}: {n} violations")
        return n == 0
    except Exception as e:
        _errors.append(f"[CRITICAL] {name}: query error - {str(e)[:200]}")
        return False

# required tables
for t in ["part", "lineitem", "orders", "partsupp", "supplier"]:
    try:
        spark.sql(f"SELECT 1 FROM {S}.{t} LIMIT 1")
    except Exception:
        _errors.append(f"[CRITICAL] missing table: {S}.{t}")

# key uniqueness and nulls
_check("part: null p_partkey", f"SELECT COUNT(*) FROM {S}.part WHERE p_partkey IS NULL")
_check("part: duplicate p_partkey", f"SELECT COUNT(*) FROM (SELECT p_partkey FROM {S}.part GROUP BY p_partkey HAVING COUNT(*)>1)")
_check("orders: null o_orderkey", f"SELECT COUNT(*) FROM {S}.orders WHERE o_orderkey IS NULL")
_check("orders: duplicate o_orderkey", f"SELECT COUNT(*) FROM (SELECT o_orderkey FROM {S}.orders GROUP BY o_orderkey HAVING COUNT(*)>1)")
_check("partsupp: null keys", f"SELECT COUNT(*) FROM {S}.partsupp WHERE ps_partkey IS NULL OR ps_suppkey IS NULL")
_check("partsupp: duplicate pairs", f"SELECT COUNT(*) FROM (SELECT ps_partkey, ps_suppkey FROM {S}.partsupp GROUP BY ps_partkey, ps_suppkey HAVING COUNT(*)>1)")
_check("lineitem: null keys", f"SELECT COUNT(*) FROM {S}.lineitem WHERE l_orderkey IS NULL OR l_linenumber IS NULL")
_check("lineitem: duplicate order-line", f"SELECT COUNT(*) FROM (SELECT l_orderkey, l_linenumber FROM {S}.lineitem GROUP BY l_orderkey, l_linenumber HAVING COUNT(*)>1)")

# referential integrity
_check("lineitem->orders orphans", f"SELECT COUNT(*) FROM {S}.lineitem l LEFT ANTI JOIN {S}.orders o ON l.l_orderkey=o.o_orderkey")
_check("lineitem->part orphans", f"SELECT COUNT(*) FROM {S}.lineitem l LEFT ANTI JOIN {S}.part p ON l.l_partkey=p.p_partkey")
_check("lineitem->partsupp orphans", f"SELECT COUNT(*) FROM {S}.lineitem l LEFT ANTI JOIN {S}.partsupp ps ON l.l_partkey=ps.ps_partkey AND l.l_suppkey=ps.ps_suppkey")
_check("partsupp->part orphans", f"SELECT COUNT(*) FROM {S}.partsupp ps LEFT ANTI JOIN {S}.part p ON ps.ps_partkey=p.p_partkey")
_check("partsupp->supplier orphans", f"SELECT COUNT(*) FROM {S}.partsupp ps LEFT ANTI JOIN {S}.supplier s ON ps.ps_suppkey=s.s_suppkey")

# data quality
_check("part: null/empty brand", f"SELECT COUNT(*) FROM {S}.part WHERE p_brand IS NULL OR TRIM(p_brand)=''")
_check("part: null/empty manufacturer", f"SELECT COUNT(*) FROM {S}.part WHERE p_mfgr IS NULL OR TRIM(p_mfgr)=''")
_check("part: non-positive retailprice", f"SELECT COUNT(*) FROM {S}.part WHERE p_retailprice IS NULL OR p_retailprice<=0")
_check("lineitem: null/non-positive quantity", f"SELECT COUNT(*) FROM {S}.lineitem WHERE l_quantity IS NULL OR l_quantity<=0")
_check("lineitem: null extendedprice", f"SELECT COUNT(*) FROM {S}.lineitem WHERE l_extendedprice IS NULL")
_check("lineitem: null discount", f"SELECT COUNT(*) FROM {S}.lineitem WHERE l_discount IS NULL")
_check("partsupp: null supplycost", f"SELECT COUNT(*) FROM {S}.partsupp WHERE ps_supplycost IS NULL")
_check("part: no supplier offers", f"SELECT COUNT(*) FROM {S}.part p WHERE NOT EXISTS (SELECT 1 FROM {S}.partsupp ps WHERE ps.ps_partkey=p.p_partkey)", critical=False)

for table in ["part", "lineitem", "orders", "partsupp"]:
    _check(f"{table}: empty input", f"SELECT CASE WHEN COUNT(*)=0 THEN 1 ELSE 0 END FROM {S}.{table}")
_check("orders: null date", f"SELECT COUNT(*) FROM {S}.orders WHERE o_orderdate IS NULL")
_check("part: null category", f"SELECT COUNT(*) FROM {S}.part WHERE p_type IS NULL OR TRIM(p_type)=''")

if _errors:
    print("INPUT VALIDATION FAILED:")
    for e in _errors:
        print(f"  {e}")
    raise RuntimeError(f"Input validation failed with {len(_errors)} errors. Gold writes aborted.")
print(f"Input validation passed. run_id={run_id}")
input_versions = {
    t: spark.sql(f"DESCRIBE HISTORY {S}.{t}").select("version")
        .orderBy("version", ascending=False).first()[0]
    for t in ["part", "lineitem", "orders", "partsupp"]
}


In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS bda_gold;

-- compute discounted revenue once, reuse for unit price and deviation
CREATE OR REPLACE TABLE bda_gold.brand_sales
USING DELTA AS
WITH enriched AS (
    SELECT
        l.l_orderkey      AS order_key,
        l.l_linenumber     AS line_number,
        l.l_partkey        AS part_key,
        l.l_suppkey        AS supplier_key,
        o.o_orderdate      AS order_date,
        CAST(date_trunc('MONTH', o.o_orderdate) AS DATE) AS month_start,
        p.p_brand          AS brand,
        p.p_mfgr           AS manufacturer,
        p.p_type           AS category,
        l.l_quantity       AS quantity,
        p.p_retailprice    AS list_unit_price,
        ps.ps_supplycost   AS supply_unit_cost,
        l.l_extendedprice * (1 - l.l_discount) AS discounted_revenue
    FROM bda_silver.lineitem l
    JOIN bda_silver.orders   o  ON l.l_orderkey = o.o_orderkey
    JOIN bda_silver.part     p  ON l.l_partkey  = p.p_partkey
    JOIN bda_silver.partsupp ps ON l.l_partkey  = ps.ps_partkey
                                AND l.l_suppkey = ps.ps_suppkey
)
SELECT
    order_key, line_number, part_key, supplier_key,
    order_date, month_start, brand, manufacturer, category,
    quantity, list_unit_price, supply_unit_cost,
    discounted_revenue AS net_revenue,
    list_unit_price - supply_unit_cost AS list_unit_margin,
    discounted_revenue / quantity AS actual_unit_price,
    ABS(discounted_revenue / quantity - list_unit_price) / list_unit_price AS price_deviation_ratio
FROM enriched;

In [ ]:
%sql
-- dense monthly calendar: all months x all brand-category pairs from product dimension
-- zero-sales months included; market share null when category revenue is zero
CREATE OR REPLACE TABLE bda_gold.brand_category_monthly
USING DELTA AS
WITH date_bounds AS (
    SELECT CAST(date_trunc('MONTH', MIN(order_date)) AS DATE) AS min_month,
           CAST(date_trunc('MONTH', MAX(order_date)) AS DATE) AS max_month
    FROM bda_gold.brand_sales
),
months AS (
    SELECT explode(sequence(min_month, max_month, INTERVAL 1 MONTH)) AS month_start
    FROM date_bounds
),
brand_categories AS (
    SELECT DISTINCT p_brand AS brand, p_type AS category
    FROM bda_silver.part
),
calendar AS (
    SELECT m.month_start, bc.brand, bc.category
    FROM months m CROSS JOIN brand_categories bc
),
actuals AS (
    SELECT month_start, category, brand, SUM(net_revenue) AS brand_revenue
    FROM bda_gold.brand_sales
    GROUP BY month_start, category, brand
),
filled AS (
    SELECT c.month_start, c.brand, c.category,
           COALESCE(a.brand_revenue, 0) AS brand_revenue
    FROM calendar c
    LEFT JOIN actuals a ON c.month_start = a.month_start
                        AND c.brand = a.brand AND c.category = a.category
)
SELECT
    month_start, category, brand, brand_revenue,
    SUM(brand_revenue) OVER (PARTITION BY month_start, category) AS category_revenue,
    CASE WHEN SUM(brand_revenue) OVER (PARTITION BY month_start, category) = 0
         THEN NULL
         ELSE brand_revenue / SUM(brand_revenue) OVER (PARTITION BY month_start, category)
    END AS market_share
FROM filled;

In [ ]:
%sql
-- one row per product, includes products with no sales
CREATE OR REPLACE TABLE bda_gold.part_list_margin
USING DELTA AS
SELECT
    p.p_partkey AS part_key,
    p.p_brand   AS brand,
    p.p_type    AS category,
    COUNT(ps.ps_suppkey) AS supplier_offer_count,
    AVG(p.p_retailprice - ps.ps_supplycost) AS average_list_unit_margin,
    MIN(p.p_retailprice - ps.ps_supplycost) AS minimum_list_unit_margin,
    MAX(p.p_retailprice - ps.ps_supplycost) AS maximum_list_unit_margin
FROM bda_silver.part p
LEFT JOIN bda_silver.partsupp ps ON p.p_partkey = ps.ps_partkey
GROUP BY p.p_partkey, p.p_brand, p.p_type;

In [ ]:
%sql
-- one row per affected product (at least one sale line with deviation > 0.15)
CREATE OR REPLACE VIEW bda_gold.price_deviation_products AS
SELECT
    part_key, brand, category,
    COUNT(*) AS anomalous_line_count,
    MAX(price_deviation_ratio) AS maximum_deviation_ratio
FROM bda_gold.brand_sales
WHERE price_deviation_ratio > 0.15
GROUP BY part_key, brand, category;

In [ ]:
%sql
SELECT COUNT(*) AS brand_sales_rows FROM bda_gold.brand_sales;
SELECT COUNT(*) AS monthly_rows FROM bda_gold.brand_category_monthly;
SELECT COUNT(*) AS part_margin_rows FROM bda_gold.part_list_margin;
SELECT COUNT(*) AS products_without_supplier
FROM bda_gold.part_list_margin WHERE supplier_offer_count = 0;
SELECT COUNT(*) AS anomalous_products FROM bda_gold.price_deviation_products;

## Q1 — Brand#32 Revenue and Category Share

Category totals include ALL brands. Brand#32 can exist in a category without sales (zero revenue). Share is null when the category has no revenue at all.

In [ ]:
%sql
-- total revenue
SELECT SUM(net_revenue) AS brand32_total_revenue
FROM bda_gold.brand_sales
WHERE brand = 'Brand#32';

-- share per category (from dense monthly, includes zero-sales)
WITH b32 AS (
    SELECT category,
        SUM(brand_revenue) AS brand32_revenue,
        SUM(category_revenue) AS category_revenue
    FROM bda_gold.brand_category_monthly
    WHERE brand = 'Brand#32'
    GROUP BY category
)
SELECT
    category,
    ROUND(brand32_revenue, 2) AS brand32_revenue,
    ROUND(category_revenue, 2) AS category_revenue,
    CASE WHEN category_revenue > 0
         THEN ROUND(brand32_revenue / category_revenue * 100, 2)
         ELSE NULL
    END AS brand32_share_pct
FROM b32
ORDER BY brand32_revenue DESC;

## Q2 — Brand#32 Average List-Price Margin

Average margin: supplier-offer average per product, then equal-product average. Products with no supplier offers have null margin (not zero).

In [ ]:
%sql
SELECT
    COUNT(*) AS product_count,
    COUNT(average_list_unit_margin) AS products_with_known_margin,
    ROUND(AVG(average_list_unit_margin), 4) AS average_list_unit_margin
FROM bda_gold.part_list_margin
WHERE brand = 'Brand#32';

## Q3 — Category Dominance and Stability

Rank brands within every category by unrounded all-time share. Stability threshold (project assumption): share range <= 5 percentage points. Insufficient data: fewer than 2 comparable months.

In [ ]:
%sql
-- uses pre-aggregated monthly table (300K rows) instead of brand_sales (30M)
WITH totals AS (
    SELECT category, brand, SUM(brand_revenue) AS brand_revenue
    FROM bda_gold.brand_category_monthly
    GROUP BY category, brand
),
shares AS (
    SELECT category, brand, brand_revenue,
        brand_revenue / NULLIF(SUM(brand_revenue) OVER (PARTITION BY category), 0) AS market_share
    FROM totals
)
SELECT
    category,
    brand,
    ROUND(market_share * 100, 4) AS market_share_pct,
    DENSE_RANK() OVER (PARTITION BY category ORDER BY market_share DESC) AS category_rank
FROM shares
ORDER BY market_share DESC, category, brand;

In [ ]:
# find all tied winners (brands with the largest all-time single-category share)
max_share = spark.sql("""
    WITH totals AS (
        SELECT category, brand, SUM(brand_revenue) AS brand_revenue
        FROM bda_gold.brand_category_monthly
        GROUP BY category, brand
    ),
    shares AS (
        SELECT category, brand,
            brand_revenue / NULLIF(SUM(brand_revenue) OVER (PARTITION BY category), 0) AS market_share
        FROM totals
    )
    SELECT MAX(market_share) AS max_share FROM shares
""").collect()[0][0]

winners = spark.sql(f"""
    WITH totals AS (
        SELECT category, brand, SUM(brand_revenue) AS brand_revenue
        FROM bda_gold.brand_category_monthly
        GROUP BY category, brand
    ),
    shares AS (
        SELECT category, brand,
            brand_revenue / NULLIF(SUM(brand_revenue) OVER (PARTITION BY category), 0) AS market_share
        FROM totals
    )
    SELECT category, brand, market_share
    FROM shares
    WHERE market_share = {max_share}
    ORDER BY category, brand
""").collect()

print("Tied winners (largest all-time single-category share):")
for w in winners:
    print(f"  {w.brand} in '{w.category}' with {w.market_share*100:.4f}% share")

# stability analysis: exclude boundary months (first and last) as potentially incomplete
# project assumption: stable means share range <= 5 percentage points
# insufficient data: fewer than 2 complete months
for w in winners:
    spark.sql(f"""
        WITH bounds AS (
            SELECT
                CAST(date_trunc('MONTH', MIN(order_date)) AS DATE) AS min_month,
                CAST(date_trunc('MONTH', MAX(order_date)) AS DATE) AS max_month
            FROM bda_gold.brand_sales
        ),
        monthly AS (
            SELECT
                m.month_start,
                m.brand,
                m.market_share,
                RANK() OVER (
                    PARTITION BY m.month_start, m.category
                    ORDER BY m.market_share DESC
                ) AS monthly_category_rank
            FROM bda_gold.brand_category_monthly m
            CROSS JOIN bounds b
            WHERE m.category = '{w.category}'
              AND m.market_share IS NOT NULL
              AND m.month_start > b.min_month
              AND m.month_start < b.max_month
        )
        SELECT
            '{w.brand}' AS brand,
            '{w.category}' AS category,
            COUNT(*) AS comparable_months,
            ROUND(MIN(market_share) * 100, 2) AS min_share_pct,
            ROUND(MAX(market_share) * 100, 2) AS max_share_pct,
            ROUND((MAX(market_share) - MIN(market_share)) * 100, 2) AS share_range_pp,
            ROUND(STDDEV(market_share) * 100, 2) AS stddev_pct,
            MIN(monthly_category_rank) AS best_monthly_rank,
            MAX(monthly_category_rank) AS worst_monthly_rank,
            CASE WHEN COUNT(*) < 2 THEN 'INSUFFICIENT DATA'
                 WHEN (MAX(market_share) - MIN(market_share)) * 100 <= 5 THEN 'STABLE'
                 ELSE 'UNSTABLE'
            END AS stability
        FROM monthly
        WHERE brand = '{w.brand}'
    """).show(truncate=False)

## Q4 — Products with Price Deviation > 15%

Affected product: at least one sale line with deviation > 0.15. Denominator: distinct sold products in the corresponding group. Groups with zero affected products are retained.

In [ ]:
%sql
-- total affected products
SELECT COUNT(*) AS anomalous_product_count
FROM bda_gold.price_deviation_products;

-- affected rate by brand
WITH sold AS (
    SELECT brand, COUNT(DISTINCT part_key) AS total_products
    FROM bda_gold.brand_sales GROUP BY brand
),
affected AS (
    SELECT brand, COUNT(DISTINCT part_key) AS affected_products
    FROM bda_gold.brand_sales WHERE price_deviation_ratio > 0.15
    GROUP BY brand
)
SELECT s.brand,
    COALESCE(a.affected_products, 0) AS affected_products,
    s.total_products,
    ROUND(COALESCE(a.affected_products, 0) / NULLIF(s.total_products, 0) * 100, 2) AS affected_pct
FROM sold s
LEFT JOIN affected a ON s.brand = a.brand
ORDER BY affected_pct DESC;

-- affected rate by brand-category
WITH sold AS (
    SELECT brand, category, COUNT(DISTINCT part_key) AS total_products
    FROM bda_gold.brand_sales GROUP BY brand, category
),
affected AS (
    SELECT brand, category, COUNT(DISTINCT part_key) AS affected_products
    FROM bda_gold.brand_sales WHERE price_deviation_ratio > 0.15
    GROUP BY brand, category
)
SELECT s.brand, s.category,
    COALESCE(a.affected_products, 0) AS affected_products,
    s.total_products,
    ROUND(COALESCE(a.affected_products, 0) / NULLIF(s.total_products, 0) * 100, 2) AS affected_pct
FROM sold s
LEFT JOIN affected a ON s.brand = a.brand AND s.category = a.category
ORDER BY affected_pct DESC;

In [ ]:
import json
from datetime import datetime

for table, version in input_versions.items():
    current = spark.sql(f"DESCRIBE HISTORY {S}.{table}").select("version").orderBy("version", ascending=False).first()[0]
    if current != version:
        raise RuntimeError(f"input changed during build: {S}.{table}; rerun Gold")

gold_version = spark.sql("DESCRIBE HISTORY bda_gold.brand_sales").select("version").orderBy("version", ascending=False).first()[0]
spark.sql("""
    CREATE TABLE IF NOT EXISTS bda_gold.run_audit (
        run_id STRING, executed_at TIMESTAMP, gold_version BIGINT,
        input_versions STRING, validation_status STRING
    ) USING DELTA
""")
audit = [(run_id, datetime.now(), gold_version, json.dumps(input_versions), "pending")]
spark.createDataFrame(audit, "run_id string, executed_at timestamp, gold_version long, input_versions string, validation_status string").write.mode("append").saveAsTable("bda_gold.run_audit")
print(f"run_id={run_id}; gold_version={gold_version}; validation=pending")


In [ ]:
%sql --name summary_counts
SELECT 'brand_sales' AS table_name, COUNT(*) AS row_count FROM bda_gold.brand_sales
UNION ALL
SELECT 'brand_category_monthly', COUNT(*) FROM bda_gold.brand_category_monthly
UNION ALL
SELECT 'part_list_margin', COUNT(*) FROM bda_gold.part_list_margin
UNION ALL
SELECT 'price_deviation_products', COUNT(*) FROM bda_gold.price_deviation_products;

-- Q1: Brand#32 total revenue
SELECT SUM(net_revenue) AS brand32_total_revenue
FROM bda_gold.brand_sales
WHERE brand = 'Brand#32';

-- Q4: anomalous product count
SELECT COUNT(*) AS anomalous_product_count
FROM bda_gold.price_deviation_products;